# 15 — Felanalys: mustasch-detektorn (binär modell)
Samma princip som `11_error_analysis_3class.ipynb`, men för den binära `mustache_detector_3.keras`. Granska felklassificerade bilder innan livetester i appen — är de genuina modellmissar, eller dolda felmärkningar i CelebA?

In [1]:
import os
import numpy as np
import tensorflow as tf

DATA_DIR = 'data/dataset_v3'

mustache_model = tf.keras.models.load_model('models/mustache_detector_3.keras')

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(178, 178),
    batch_size=32,
    label_mode='binary',
    shuffle=False
)

class_names = val_ds.class_names  # alfabetisk: ['clean', 'mustache']
file_paths = val_ds.file_paths

y_true, y_pred, probs = [], [], []
for images, labels in val_ds:
    preds = mustache_model.predict(images, verbose=0).flatten()
    y_true.extend(labels.numpy().flatten().astype(int))
    y_pred.extend((preds >= 0.5).astype(int))
    probs.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
probs = np.array(probs)

assert len(file_paths) == len(y_true)
print(f'{len(file_paths)} valideringsbilder, {class_names}')

Found 11924 files belonging to 2 classes.
Using 2384 files for validation.
2384 valideringsbilder, ['clean', 'mustache']


2026-06-26 18:30:44.141624: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Visa felklassificerade bilder
Byt `TRUE_LABEL` mellan `'mustache'` (missade mustascher) och `'clean'` (falsklarm) för att granska båda grupperna.

In [3]:
import matplotlib.pyplot as plt
from PIL import Image

TRUE_LABEL = 'clean'  # eller 'clean'
true_idx = class_names.index(TRUE_LABEL)
wrong_idx = 1 - true_idx

matching = [
    i for i in range(len(y_true))
    if y_true[i] == true_idx and y_pred[i] == wrong_idx
]

print(f'{len(matching)} felklassificerade bilder med sann etikett "{TRUE_LABEL}".')

if len(matching) == 0:
    print('Inga bilder att visa.')
else:
    n = min(len(matching), 40)
    cols = 5
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
    axes = np.array(axes).flatten()

    for ax, i in zip(axes, matching[:n]):
        img = Image.open(file_paths[i])
        ax.imshow(img)
        ax.set_title(f'{os.path.basename(file_paths[i])}\nprob={probs[i]:.3f}', fontsize=8)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

0 felklassificerade bilder med sann etikett "clean".
Inga bilder att visa.
